In [3]:
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import kagglehub as kg
from torch._C import dtype



In [4]:
kg.login()

In [5]:
from pathlib import Path

path = kg.dataset_download("bhatvikas/movielens-100k-dataset")
print(f"Dataset path : " , path)
# path = kagglehub.dataset_download("bhatvikas/movielens-100k-dataset")
# Путь к основным файлам
data_dir = Path(path) / "ml-100k"

ratings = pd.read_csv(
    data_dir/"u.data",
    sep="\t",
    names=[
        "user_id",
        "item_id",
        "rating",
        "timestamp",

    ]
)


Dataset path :  C:\Users\Legion5\.cache\kagglehub\datasets\bhatvikas\movielens-100k-dataset\versions\1


In [6]:
print(ratings.head())
print(f"Shape : {ratings.shape}  and Movies : {ratings.item_id.nunique()}")


   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
Shape : (100000, 4)  and Movies : 1682


In [7]:
movies = pd.read_csv(
    data_dir/"u.item",
    sep="|",
    encoding="latin-1",
        names=[
        "item_id",
        "title",
        "release_date",
        "video_release_date",
        "imdb_url"
    ] +[f"genre_{i}" for i in range(19)]
)

print(f"Movies : {movies[['item_id' , 'title']]}")
print(movies.head())

Movies :       item_id                                      title
0           1                           Toy Story (1995)
1           2                           GoldenEye (1995)
2           3                          Four Rooms (1995)
3           4                          Get Shorty (1995)
4           5                             Copycat (1995)
...       ...                                        ...
1677     1678                          Mat' i syn (1997)
1678     1679                           B. Monkey (1998)
1679     1680                       Sliding Doors (1998)
1680     1681                        You So Crazy (1994)
1681     1682  Scream of Stone (Schrei aus Stein) (1991)

[1682 rows x 2 columns]
   item_id              title release_date  video_release_date  \
0        1   Toy Story (1995)  01-Jan-1995                 NaN   
1        2   GoldenEye (1995)  01-Jan-1995                 NaN   
2        3  Four Rooms (1995)  01-Jan-1995                 NaN   
3        4  Get Sh

In [8]:
user_to_idx = {
    uid: i
    for i , uid in enumerate(ratings['user_id'].unique())
}

movie_to_idx = {
    mid:i
    for i , mid in enumerate(ratings['item_id'].unique())
}

idx_to_movie = {
    i : mid
    for mid , i in movie_to_idx.items()
}

ratings['user_idx'] = ratings['user_id'].map(user_to_idx)
ratings['movie_idx'] = ratings['item_id'].map(movie_to_idx)

print(f"Users : {len(user_to_idx)}\nMovie: {len(movie_to_idx)}\nMean rattin in dataset : {ratings['rating'].mean()}")


Users : 943
Movie: 1682
Mean rattin in dataset : 3.52986


In [9]:
from torch.utils.data import DataLoader , Dataset


class MovieDataset(Dataset):
    def __init__(self , df):
        self.users = torch.tensor(df["user_idx"].values , dtype=torch.long)
        self.movie = torch.tensor(df["movie_idx"].values , dtype=torch.long)
        self.rating = torch.tensor(df["rating"].values , dtype=torch.float32)


    def __len__(self):
        return len(self.rating)


    def __getitem__(self, index):
        return self.users[index], self.movie[index],self.rating[index]



In [10]:
training_df = ratings.sample(frac=0.8 , random_state=42)

val_df = ratings.drop(training_df.index)

training_loader = DataLoader(MovieDataset(training_df) , shuffle=True,batch_size=256)
value_loader = DataLoader(MovieDataset(val_df) , shuffle=False,batch_size=256)

print(f"Movie training dataloader : {training_df.head()}")
print("=="*50)
print(f"Movue value training loader : {val_df.head()}")


Movie training dataloader :        user_id  item_id  rating  timestamp  user_idx  movie_idx
75721      877      381       4  882677345       873        377
80184      815      602       3  878694269       808        601
19864       94      431       4  891721716        90        354
76699      416      875       2  876696938       409        570
92991      500      182       2  883873556       496        356
Movue value training loader :     user_id  item_id  rating  timestamp  user_idx  movie_idx
2        22      377       1  878887116         2          2
5       298      474       4  884182806         5          5
15      303      785       3  879485318        15         15
18      291     1042       4  874834944        18         18
20      119      392       4  886176814        20         20


In [11]:
class MatrixFactorization(nn.Module):
    def __init__(self , num_users, num_movies, global_mean=3.5, embedding_dim=32):
        super().__init__()

        self.g_mean = nn.Parameter(
            torch.tensor(global_mean) ,
            requires_grad=False
        )

        # Embedding
        self.user_emb = nn.Embedding(num_users , embedding_dim)
        self.movie_emb = nn.Embedding(
            num_movies , embedding_dim
        )

        self.user_bias = nn.Embedding(num_users , 1)
        self.movie_bias = nn.Embedding(num_movies , 1)

        nn.init.normal_(self.user_emb.weight , std=0.1) #nn.init.normal_
        nn.init.normal_(self.movie_emb.weight ,std=0.1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self , u_idx , m_idx):
        user = self.user_emb(u_idx)
        movie = self.movie_emb(m_idx)
        interaction = (user*movie).sum(dim=1,keepdim=True)
        prediction = (
            self.g_mean + self.user_bias(u_idx) + self.movie_bias(m_idx) + interaction
        )

        return prediction.squeeze()


In [12]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    embedding_dim : int = 32
    lr: float = 5e-3
    weight_decay: float =1e-4
    epochs: int = 20
    batch_size: int =1024
    grad_clip : float = 1.0
    patience: int = 3
    seed: int = 42
    out_dir: str = "runs/mf_v1"

def seed_everything(seed: int) -> None :
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

class Trainer:
    def __init__(self , model , cfg:Config , training_loader , value_loader , device ):
        self.model = model.to(device)
        self.cfg = cfg
        self.training_loader = training_loader
        self.value_loader = value_loader
        self.device = device
        self.criterion = nn.MSELoss()
        self.optim = torch.optim.Adam(
            model.parameters(), lr=cfg.lr , weight_decay=cfg.weight_decay
        )
        self.sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optim , factor=0.5, patience=1
        )
        self.out = Path(cfg.out_dir)
        self.out.mkdir(parents=True,exist_ok=True)
        (self.out / "config.json").write_text(
            json.dumps(asdict(cfg), indent=2)
        )
        self.base_rmse = float('inf')


    def _step(self , u , m , r):
        self.optim.zero_grad(set_to_none=True)
        loss = self.criterion(self.model(u , m ),r)
        loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip)
        self.optim.step()

        return loss.item()

    def train_epoch(self) -> float:
        self.model.train()
        total , l = 0.0 , 0
        for u , m ,r in self.training_loader:
            u , m , r = (x.to(self.device , non_blocking=True) for x in (u,m,r))
            total += self._step(u , m , r) *len(r)
            l += len(r)
        return total / l

    @torch.no_grad()
    def evaluate(self ) -> float:
        self.model.eval()
        total , n = 0.0 , 0
        for u , m , r in self.value_loader:
            u , m , r = (x.to(self.device , non_blocking=True) for x in (u,m,r))
            total += self.criterion(
                self.model(u , m) , r
            ).item() * len(r)
            n += len(r)

        return float(np.sqrt(total / n))

    def fit(self):
        bad_epoch = 0

        for epoch in range(1 , self.cfg.epochs+1):
            train_mse = self.train_epoch()
            val_rmse = self.evaluate()
            self.sched.step(
                val_rmse
            )
            print(f"epoch : {epoch:02d}| train_mse : {train_mse:.4f}|"
                  f"val rmse : {val_rmse:.4f} | lr : {self.optim.param_groups[0]['lr']:.2e}")

            if val_rmse < self.base_rmse:
                self.base_rmse , bad_epoch = val_rmse , 0
                torch.save(self.model.state_dict(), self.out / "best.pt")
            else :
                bad_epoch+=1
                if bad_epoch >= self.cfg.patience:
                    print("early stopping")
                    break



In [13]:
cfg = Config()
seed_everything(cfg.seed)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
global_mean = float(training_df["rating"].mean())

user_len = len(user_to_idx)
movie_len = len(movie_to_idx)
model = MatrixFactorization(
    user_len , movie_len,
    embedding_dim=cfg.embedding_dim,
    global_mean=global_mean
)

trainer = Trainer(model , cfg , training_loader , value_loader, device)
trainer.fit()

epoch : 01| train_mse : 1.0768|val rmse : 0.9780 | lr : 5.00e-03
epoch : 02| train_mse : 0.8128|val rmse : 0.9236 | lr : 5.00e-03
epoch : 03| train_mse : 0.6523|val rmse : 0.9216 | lr : 5.00e-03
epoch : 04| train_mse : 0.5361|val rmse : 0.9318 | lr : 5.00e-03
epoch : 05| train_mse : 0.4624|val rmse : 0.9440 | lr : 2.50e-03
epoch : 06| train_mse : 0.3782|val rmse : 0.9488 | lr : 2.50e-03
early stopping


In [16]:
seen_by_user = ratings.groupby("user_id")["item_id"].apply(set).to_dict()
titles = movies.set_index("item_id")["title"]

@torch.no_grad()
def recommendation(user_id , top_k=5):
    model.eval()

    seen = seen_by_user.get(user_id , set())
    cand = [idx for mid , idx in movie_to_idx.items() if mid not in seen]

    m = torch.tensor(cand, dtype=torch.long , device=device)
    u = torch.full_like(m , user_to_idx[user_id])
    scores = model(u , m)

    top = torch.topk(scores , k=min(top_k , len(cand)))

    return [
        (int(idx_to_movie[cand[i]]) , titles[idx_to_movie[i]] , round(s , 3))
        for i , s in zip(top.indices.tolist() , top.values.tolist())
    ]


In [18]:
print(recommendation(8 , 5))

[(98, 'Jaws (1975)', 5.145), (173, 'English Patient, The (1996)', 4.909), (64, 'Cape Fear (1991)', 4.887), (48, 'River Wild, The (1994)', 4.878), (178, 'Chain Reaction (1996)', 4.823)]
